# 04 · Qualidade de Dados

**Objetivo:** verificar, atributo por atributo, a qualidade dos dados capturados do PNCP e
registrar como cada problema encontrado foi tratado no pipeline.

Foi avaliado:

| Dimensão | Pergunta que responde |
|---|---|
| Completude | Existem valores nulos ou vazios? Em que proporção? |
| Consistência | Os valores seguem o padrão esperado? |
| Unicidade | Existem duplicatas onde não deveria haver? |
| Acurácia | Os valores fazem sentido no contexto? |
| Outliers | Existem valores extremos que distorcem as análises? |

A verificação é feita sobre a Bronze e sobre a Silver, pra mostrar o efeito das transformações.

In [0]:
CATALOGO = "workspace"
BRONZE = f"{CATALOGO}.bronze.pncp_contratacoes"
SILVER = f"{CATALOGO}.silver.contratacoes"

from pyspark.sql import functions as F

bronze = spark.table(BRONZE)
silver = spark.table(SILVER)

print(f"Bronze: {bronze.count()} linhas | Silver: {silver.count()} linhas")

Bronze: 17432 linhas | Silver: 17432 linhas


## 1. Completude — nulos por atributo

Percorre as colunas da Silver e mede a proporção de nulos em cada uma.

In [0]:
colunas = [c for c in silver.columns if not c.startswith("_")]
total_linhas = silver.count()

agregacoes = []
for coluna in colunas:
    agregacoes.append(F.sum(F.col(coluna).isNull().cast("int")).alias(coluna))

nulos = silver.agg(*agregacoes).collect()[0].asDict()

completude = spark.createDataFrame(
    [(c, nulos[c], round(100.0 * nulos[c] / total_linhas, 2)) for c in colunas],
    "atributo STRING, nulos INT, pct_nulos DOUBLE",
).orderBy(F.col("pct_nulos").desc())

display(completude)

atributo,nulos,pct_nulos
data_abertura_proposta,7693,44.13
data_encerramento_proposta,7693,44.13
prazo_proposta_dias,7693,44.13
prazo_proposta_invalido,7693,44.13
economia_valor,7440,42.68
razao_homologado_estimado,7440,42.68
economia_pct,7440,42.68
valor_total_homologado,7242,41.54
tem_homologado,7242,41.54
numero_controle_pncp,0,0.0


Leitura esperada: "valor_total_homologado" e os campos derivados dele ("economia_valor,
"economia_pct") concentram os nulos, porque muitas contratações ainda não foram concluídas ou
não tiveram o resultado atualizado no PNCP pelo órgão.

Tratamento adotado: os nulos foram preservados, não preenchidos. Substituir por zero
ia criar uma economia de 100% em contratações em andamento, distorcendo a análise. Por isso a
Silver marca a situação na flag tem_homologado e as consultas filtram por ela.

In [0]:
# Campos vazios (string em branco) também contam como incompletude, mas não são nulos
display(spark.sql(f"""
SELECT
  SUM(CASE WHEN TRIM(COALESCE(objeto_compra, '')) = ''      THEN 1 ELSE 0 END) AS objeto_vazio,
  SUM(CASE WHEN TRIM(COALESCE(orgao_razao_social, '')) = '' THEN 1 ELSE 0 END) AS razao_social_vazia,
  SUM(CASE WHEN TRIM(COALESCE(municipio_nome, '')) = ''     THEN 1 ELSE 0 END) AS municipio_vazio
FROM {SILVER}
"""))

objeto_vazio,razao_social_vazia,municipio_vazio
0,0,0


## 2. Unicidade — duplicatas

A chave natural é `numero_controle_pncp`. Duplicatas podem surgir por sobreposição entre
janelas de datas ou por execuções repetidas da coleta.

In [0]:
display(spark.sql(f"""
SELECT
  'Bronze' AS camada,
  COUNT(*)                                  AS linhas,
  COUNT(DISTINCT numeroControlePNCP)        AS chaves_distintas,
  COUNT(*) - COUNT(DISTINCT numeroControlePNCP) AS duplicatas
FROM {BRONZE}
UNION ALL
SELECT
  'Silver' AS camada,
  COUNT(*)                                   AS linhas,
  COUNT(DISTINCT numero_controle_pncp)       AS chaves_distintas,
  COUNT(*) - COUNT(DISTINCT numero_controle_pncp) AS duplicatas
FROM {SILVER}
"""))

camada,linhas,chaves_distintas,duplicatas
Bronze,17432,17432,0
Silver,17432,17432,0


Tratamento adotado: a Silver aplica "ROW_NUMBER" particionado por "numero_controle_pncp",
ordenando por "data_atualizacao_global" decrescente, e mantém apenas a versão mais recente.
A Silver deve apresentar zero duplicatas independentemente do que a Bronze traga.

## 3. Consistência — formatos e domínios

Verifica se os valores respeitam o formato e o conjunto de valores esperados.

In [0]:
display(spark.sql(f"""
SELECT
  SUM(CASE WHEN orgao_cnpj IS NOT NULL AND NOT orgao_cnpj RLIKE '^[0-9]{{14}}$' THEN 1 ELSE 0 END)
    AS cnpj_fora_de_14_digitos,
  SUM(CASE WHEN municipio_codigo_ibge IS NOT NULL AND NOT municipio_codigo_ibge RLIKE '^[0-9]{{7}}$' THEN 1 ELSE 0 END)
    AS ibge_fora_de_7_digitos,
  SUM(CASE WHEN uf_sigla <> 'PA' THEN 1 ELSE 0 END)
    AS uf_fora_do_recorte,
  SUM(CASE WHEN modalidade_id NOT BETWEEN 1 AND 13 THEN 1 ELSE 0 END)
    AS modalidade_fora_do_dominio,
  SUM(CASE WHEN orgao_esfera_id NOT IN ('F','E','M','D') THEN 1 ELSE 0 END)
    AS esfera_fora_do_dominio,
  SUM(CASE WHEN orgao_poder_id NOT IN ('E','L','J') THEN 1 ELSE 0 END)
    AS poder_fora_do_dominio
FROM {SILVER}
"""))

cnpj_fora_de_14_digitos,ibge_fora_de_7_digitos,uf_fora_do_recorte,modalidade_fora_do_dominio,esfera_fora_do_dominio,poder_fora_do_dominio
0,0,0,0,198,7951


In [0]:
# Domínio observado dos campos categóricos: confirma que as categorias são as esperadas
for coluna in ["situacao_compra_nome", "modo_disputa_nome", "tipo_instrumento_nome", "desfecho"]:
    print(f"\n--- {coluna} ---")
    display(
        silver.groupBy(coluna).count().orderBy(F.col("count").desc())
    )


--- situacao_compra_nome ---


situacao_compra_nome,count
Divulgada no PNCP,17038
Suspensa,170
Revogada,143
Anulada,81



--- modo_disputa_nome ---


modo_disputa_nome,count
Não se aplica,7971
Aberto,5573
Dispensa Com Disputa,2301
Aberto-Fechado,1453
Fechado,122
Fechado-Aberto,12



--- tipo_instrumento_nome ---


tipo_instrumento_nome,count
Ato que autoriza a Contratação Direta,7693
Edital,7183
Aviso de Contratação Direta,2301
Edital de Chamamento Público,255



--- desfecho ---


desfecho,count
homologada,10063
sem_homologacao,6975
frustrada,224
suspensa,170


In [0]:
# Consistência de datas: a ordem cronológica entre os eventos deve ser respeitada
display(spark.sql(f"""
SELECT
  SUM(CASE WHEN data_encerramento_proposta < data_abertura_proposta THEN 1 ELSE 0 END)
    AS encerramento_antes_da_abertura,
  SUM(CASE WHEN data_publicacao_pncp > data_atualizacao_global THEN 1 ELSE 0 END)
    AS publicacao_depois_da_atualizacao,
  MIN(data_publicacao) AS primeira_publicacao,
  MAX(data_publicacao) AS ultima_publicacao
FROM {SILVER}
"""))

encerramento_antes_da_abertura,publicacao_depois_da_atualizacao,primeira_publicacao,ultima_publicacao
0,0,2026-01-02,2026-08-31


## 4. Acurácia — os valores fazem sentido?

Um detalhe que foi percebido é que o valor homologado é igual ao estimado

In [0]:
display(spark.sql(f"""
SELECT
  COUNT(*)                                                                      AS total,
  SUM(CASE WHEN valor_total_estimado = 0 THEN 1 ELSE 0 END)                     AS estimado_zerado,
  SUM(CASE WHEN valor_total_estimado < 0 THEN 1 ELSE 0 END)                     AS estimado_negativo,
  SUM(CASE WHEN valor_total_homologado < 0 THEN 1 ELSE 0 END)                   AS homologado_negativo,
  SUM(CASE WHEN NOT tem_homologado THEN 1 ELSE 0 END)                           AS sem_homologado,
  SUM(CASE WHEN estimado_valido AND tem_homologado THEN 1 ELSE 0 END)           AS calculaveis,
  SUM(CASE WHEN estimado_valido AND tem_homologado
                AND valor_total_homologado = valor_total_estimado THEN 1 ELSE 0 END) AS valor_identico
FROM {SILVER}
"""))

total,estimado_zerado,estimado_negativo,homologado_negativo,sem_homologado,calculaveis,valor_identico
17432,1002,0,0,99,9992,4730


In [0]:
# Proporção de valores idênticos entre os casos calculáveis
display(spark.sql(f"""
SELECT
  COUNT(*) AS calculaveis,
  SUM(CASE WHEN valor_total_homologado = valor_total_estimado THEN 1 ELSE 0 END) AS identicos,
  ROUND(100.0 * SUM(CASE WHEN valor_total_homologado = valor_total_estimado THEN 1 ELSE 0 END) / COUNT(*), 1)
    AS pct_identicos
FROM {SILVER}
WHERE estimado_valido AND tem_homologado
"""))

calculaveis,identicos,pct_identicos
9992,4730,47.3


Interpretação: quando o valor homologado é exatamente igual ao estimado em tantos casos, dificilmente isso significa que a licitação fechou no preço previsto. Provavelmente o órgão repetiu o valor       
estimado ao preencher o campo, sem registrar o que de fato saiu da disputa. Então é possível que o valor homologado esteja incorreto em vários casos.

Tratamento adotado: como os dados são legítimos e muitos deles podem estar corretos, as analises econômicas são feitas tanto com quanto sem eles, vendo se ouve uma diferença grande nos resultados.

In [0]:
# Prazo de propostas: valores negativos são impossíveis; prazos muito longos merecem atenção
display(spark.sql(f"""
SELECT
  SUM(CASE WHEN prazo_proposta_dias < 0   THEN 1 ELSE 0 END) AS prazo_negativo,
  SUM(CASE WHEN prazo_proposta_dias = 0   THEN 1 ELSE 0 END) AS prazo_zero,
  SUM(CASE WHEN prazo_proposta_dias > 180 THEN 1 ELSE 0 END) AS prazo_acima_180_dias,
  ROUND(AVG(prazo_proposta_dias), 1)                         AS prazo_medio,
  MAX(prazo_proposta_dias)                                   AS prazo_maximo
FROM {SILVER}
WHERE prazo_proposta_dias IS NOT NULL
"""))

prazo_negativo,prazo_zero,prazo_acima_180_dias,prazo_medio,prazo_maximo
0,1133,121,20.9,9131


## 5. Outliers — valores extremos

Distribuição dos valores e da economia percentual, para identificar o que pode distorcer médias.

In [0]:
display(spark.sql(f"""
SELECT
  'valor_total_estimado' AS metrica,
  ROUND(MIN(valor_total_estimado), 2)                          AS minimo,
  ROUND(percentile(valor_total_estimado, 0.25), 2)             AS q1,
  ROUND(percentile(valor_total_estimado, 0.50), 2)             AS mediana,
  ROUND(percentile(valor_total_estimado, 0.75), 2)             AS q3,
  ROUND(percentile(valor_total_estimado, 0.99), 2)             AS p99,
  ROUND(MAX(valor_total_estimado), 2)                          AS maximo,
  ROUND(AVG(valor_total_estimado), 2)                          AS media
FROM {SILVER}
WHERE estimado_valido
UNION ALL
SELECT
  'economia_pct' AS metrica,
  ROUND(MIN(economia_pct), 2),
  ROUND(percentile(economia_pct, 0.25), 2),
  ROUND(percentile(economia_pct, 0.50), 2),
  ROUND(percentile(economia_pct, 0.75), 2),
  ROUND(percentile(economia_pct, 0.99), 2),
  ROUND(MAX(economia_pct), 2),
  ROUND(AVG(economia_pct), 2)
FROM {SILVER}
WHERE economia_pct IS NOT NULL
"""))

metrica,minimo,q1,mediana,q3,p99,maximo,media
valor_total_estimado,0.01,8640.28,88709.77,916787.8,3.289492166E7,3.6E11,2.628667983E7
economia_pct,-9.99999E7,0.0,0.1,16.8,91.2,100.0,-14080.67


Leitura esperada: a média do valor estimado deve ficar bem acima da mediana, sinal de
distribuição assimétrica com poucas contratações de valor muito alto. Na economia percentual,
valores extremamente negativos aparecem quando o estimado foi preenchido com um valor
simbólico (por exemplo R$ 1,00), fazendo o homologado parecer centenas de vezes maior.

In [0]:
display(spark.sql(f"""
SELECT numero_controle_pncp, municipio_nome, modalidade_nome,
       valor_total_estimado, valor_total_homologado, economia_pct
FROM {SILVER}
WHERE economia_pct IS NOT NULL
ORDER BY economia_pct ASC
LIMIT 10
"""))

numero_controle_pncp,municipio_nome,modalidade_nome,valor_total_estimado,valor_total_homologado,economia_pct
04976700000177-1-000120/2025,Belém,Concorrência - Presencial,1.00,1000000.00,-9.99999E7
04873592000107-1-000144/2026,Bragança,Pregão - Eletrônico,10.00,3989397.82,-3.98938782E7
05837521000111-1-000053/2026,Belém,Inexigibilidade,4200.00,17640000.00,-419900.0
00368105000106-1-000102/2026,Belém,Inexigibilidade,1100.00,1210000.00,-109900.0
50448324000165-1-000025/2025,Belém,Concorrência - Eletrônica,1267218.30,1267218000.00,-99900.0
05077102000129-1-000033/2026,Itupiranga,Credenciamento,7700.00,7700000.00,-99900.0
50448324000165-1-000012/2025,Belém,Concorrência - Eletrônica,3664761.99,3600000000.00,-98132.8
00394411000109-1-000069/2026,Belém,Credenciamento,58833633.74,19928816797.65,-33773.2
50448324000165-1-000004/2026,Belém,Pregão - Eletrônico,22018600.89,5239741500.00,-23696.9
05054903000179-1-000010/2026,Belém,Dispensa,61537.49,4122966.94,-6599.9


Tratamento adotado: nenhum outlier foi removido. Em vez de definir um limite arbitrário de
corte, as análises de economia usam mediana em lugar de média, medida naturalmente robusta
a valores extremos, e exigem um mínimo de contratações por grupo (`HAVING COUNT(*) >= 30`)
para evitar conclusões baseadas em amostras pequenas.

## Resumo dos problemas e tratamentos

| # | Problema detectado | Dimensão | Tratamento adotado |
|---|---|---|---|
| 1 | Campos aninhados em JSON (`orgaoEntidade`, `unidadeOrgao`, `amparoLegal`) | Consistência | Achatados em colunas próprias na Silver |
| 2 | Datas e valores como texto | Consistência | Convertidos para `timestamp` e `decimal(18,2)` |
| 3 | `valor_total_homologado` ausente em parte dos registros | Completude | Preservado como nulo; marcado em `tem_homologado`; filtrado nas análises |
| 4 | `valor_total_estimado` igual a zero | Acurácia | Marcado em `estimado_valido`; excluído dos cálculos de economia |
| 5 | Homologado idêntico ao estimado em proporção alta | Acurácia | Marcado em `valor_identico`; análises reportadas com e sem esses casos |
| 6 | Economia percentual com valores extremos | Outliers | Uso de mediana e mínimo de 30 contratações por grupo |
| 7 | Risco de duplicatas entre janelas e execuções da coleta | Unicidade | Deduplicação por `numero_controle_pncp`, mantendo a versão mais recente |
| 8 | Colunas integralmente nulas (`orgaoSubRogado`, `unidadeSubRogada`) | Completude | Descartadas na Silver por ausência de conteúdo |